##### Why aiohttp.ClientSession vs requests vs httpx

1) Concurrency & Throughput

- With asyncio, you can schedule hundreds/thousands of concurrent fetches without spawning threads.
Async I/O naturally overlaps waiting for network with other tasks, giving high throughput on I/O-bound workloads.

- Requests is synchronous and thread-based for concurrency. For a handful of URLs, it’s fine and simple.
For many URLs or low-latency services, aiohttp’s single-threaded async model is faster, more memory-efficient, and easier to scale without thread contention.

- If you need HTTP/2 support or a unified API for sync/async, httpx is attractive.
aiohttp is still a battle-tested workhorse with strong streaming performance and ecosystem integration; many high-throughput Python services rely on it.

2) Connection Reuse & Pooling

A single ClientSession maintains a pool of connections per host, reducing TLS handshakes and latency.
Contrast: creating a new session/socket per request (e.g., naïve requests.get in a loop) causes handshake overhead and can hit OS limits under load.

3) Fine-grained Timeouts & Connectors

aiohttp has granular timeout control and custom TCPConnector options (limit total connections, per-host limits, SSL options, DNS cache TTL).
This is critical for stability under high concurrency.

4) Task Cancellation & Reliability

Async tasks can be cancelled cleanly (e.g., shutdown, time budget exceeded). The context manager ensures sockets are released.


##### Semaphore

A semaphore is an async concurrency‑control tool that limits how many tasks can run a specific block of code at the same time.
When scraping or calling many URLs concurrently (e.g., using aiohttp), without a semaphore you might accidentally start hundreds or thousands of parallel requests. That causes problems:
- 1️⃣ Protect your machine from overload
- 2️⃣ Avoid overwhelming the target server
- 3️⃣ Avoid exhausting aiohttp’s internal connection pool



#### “85% valid citations” practically mean?
When evaluating 100 queries:

At least 85 of the answers include at least one valid citation.
Remaining 15 may have:

missing citations
wrong section cited
hallucinated pages
mismatched spans



This ensures:

high grounding
low hallucination
traceability
search quality

##### 1. RAG system must output answer as well as citations

{
  "answer": "...",
  "citations": [
    {
      "chunk_id": "stripe_payments_12",
      "url": "https://docs.stripe.com/payments/accept-a-payment#confirm",
      "text_span": "The PaymentIntent must be confirmed server-side."
    }
  ]
}

##### Build a “Citation Validator” component
    A. Existence check
    B. Span Fidelity Check / Cosine similarity check > 75%
    C. No ghost citations

No ghost citations

- URL does not exist
- chunk_id unknown
- citation refers to a section not used in retrieval
